In [1]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV,StratifiedKFold
import pandas as pd
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import tree
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../Data Files/Processed Files/credit_card_fraud_clean.csv')
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2026, stratify = y)

In [4]:
sm = SMOTE(random_state = 2026)

X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [5]:
naive_bayes_model = GaussianNB()
decision_tree_model = tree.DecisionTreeClassifier(random_state = 2026)
gradient_boosting_model = HistGradientBoostingClassifier(random_state = 2026, max_iter = 100)
random_forest_model = RandomForestClassifier(random_state = 2026, n_estimators = 100)
adaboost_model = AdaBoostClassifier(random_state = 2026, n_estimators = 100)
xgb_model = XGBClassifier(
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum(),
    random_state = 2026,
    n_estimators = 100
)

models = [naive_bayes_model, decision_tree_model, gradient_boosting_model, random_forest_model, adaboost_model, xgb_model]

In [6]:
metrics = {}
def model_train_and_evaluate(model, X_train, y_train, X_test, y_test):
    model_name = model.__class__.__name__
    mlflow.set_experiment("credit_card_fraud_detection")
    with mlflow.start_run(run_name = model_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)

        precision = report['1']['precision']
        recall = report['1']['recall']
        f1_score = report['1']['f1-score']
        accuracy = accuracy_score(y_test, y_pred)

        metrics[model_name] = {
            'precision': precision,
            'recall': recall,
            'f1-score': f1_score,
            'accuracy': accuracy
        }  

        mlflow.log_params(model.get_params())
        mlflow.log_metrics({
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "accuracy": accuracy
            })

        signature = infer_signature(X_train, y_pred)

        mlflow.sklearn.log_model(
            sk_model = model,
            artifact_path = model_name,
            signature = signature,
            input_example = X_train.head(5)
            )
    return model
        
   

In [7]:
for model in models:
    print(f"Training and evaluating {model.__class__.__name__}")
    model_train_and_evaluate(model, X_train_res, y_train_res, X_test, y_test)
    print(f"Metrics for {model.__class__.__name__}: {metrics[model.__class__.__name__]}")

2026/05/15 00:03:49 INFO mlflow.tracking.fluent: Experiment with name 'credit_card_fraud_detection' does not exist. Creating a new experiment.


Training and evaluating GaussianNB


2026/05/15 00:04:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Metrics for GaussianNB: {'precision': 0.12794033275960986, 'recall': 0.6264044943820225, 'f1-score': 0.21248213434969032, 'accuracy': 0.975663260799152}
Training and evaluating DecisionTreeClassifier


2026/05/15 00:04:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

Metrics for RandomForestClassifier: {'precision': 0.8198198198198198, 'recall': 0.7668539325842697, 'f1-score': 0.7924528301886793, 'accuracy': 0.997894643856188}
Training and evaluating AdaBoostClassifier


2026/05/15 00:12:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Metrics for AdaBoostClassifier: {'precision': 0.07617695765318193, 'recall': 0.9044943820224719, 'f1-score': 0.14051931049530875, 'accuracy': 0.9420070080386326}
Training and evaluating XGBClassifier


2026/05/15 00:13:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Metrics for XGBClassifier: {'precision': 0.26627218934911245, 'recall': 0.8848314606741573, 'f1-score': 0.4093567251461988, 'accuracy': 0.9866170018550691}


In [8]:
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)

                                precision    recall  f1-score  accuracy
GaussianNB                       0.127940  0.626404  0.212482  0.975663
DecisionTreeClassifier           0.572368  0.733146  0.642857  0.995730
HistGradientBoostingClassifier   0.493769  0.890449  0.635271  0.994641
RandomForestClassifier           0.819820  0.766854  0.792453  0.997895
AdaBoostClassifier               0.076177  0.904494  0.140519  0.942007
XGBClassifier                    0.266272  0.884831  0.409357  0.986617


In [17]:
X_small = X.head(1000)
X_small['target'] = y[:1000]

X_small.to_csv('../Testing/sample.csv', index=False)